##Naseem Saleh
##INST414
##Module 3 Assignment¶
##October 17, 2025

dataset source:
https://www.kaggle.com/datasets/tamber/steam-video-games


"This dataset is a list of user behaviors, with columns: user-id, game-title, behavior-name, value. The behaviors included are 'purchase' and 'play'. The value indicates the degree to which the behavior was performed - in the case of 'purchase' the value is always 1, and in the case of 'play' the value represents the number of hours the user has played the game."

Question - What games are similar? Going by who plays what game and what other games they also play, and maybe how much time they played, we try to find out which games people might also enjoy because they play a certain game.

In [75]:
import pandas as pd

from sklearn.metrics import DistanceMetric, pairwise_distances

import matplotlib.pyplot as plt

In [38]:
#Loading the csv into df
steam_games_df = pd.read_csv("steam-200k.csv")

#Checking the first 5 rows of data
steam_games_df.head()

,151603712,The Elder Scrolls V Skyrim,purchase,1.0,0
0,151603712,The Elder Scrolls V Skyrim,play,273.0,0
1,151603712,Fallout 4,purchase,1.0,0
2,151603712,Fallout 4,play,87.0,0
3,151603712,Spore,purchase,1.0,0
4,151603712,Spore,play,14.9,0


In [39]:
#Fixing the column names:   #Don't need to add the original column names as real data? for this particular question?

col_names_dict = {"151603712": "User_ID", "The Elder Scrolls V Skyrim": "Game_Title", "purchase": "Behavior_Name", "1.0": "Hours_Played"}

steam_games_df = steam_games_df.rename(columns = col_names_dict)

steam_games_df.head()

,User_ID,Game_Title,Behavior_Name,Hours_Played,0
0,151603712,The Elder Scrolls V Skyrim,play,273.0,0
1,151603712,Fallout 4,purchase,1.0,0
2,151603712,Fallout 4,play,87.0,0
3,151603712,Spore,purchase,1.0,0
4,151603712,Spore,play,14.9,0


In [40]:
#Checking data types
print(steam_games_df.dtypes)

User_ID            int64
Game_Title        object
Behavior_Name     object
Hours_Played     float64
0                  int64
dtype: object


In [41]:
steam_games_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199999 entries, 0 to 199998
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   User_ID        199999 non-null  int64  
 1   Game_Title     199999 non-null  object 
 2   Behavior_Name  199999 non-null  object 
 3   Hours_Played   199999 non-null  float64
 4   0              199999 non-null  int64  
dtypes: float64(1), int64(2), object(2)
memory usage: 7.6+ MB


In [42]:
#Checking for null values
steam_games_df.isnull().sum()

User_ID          0
Game_Title       0
Behavior_Name    0
Hours_Played     0
0                0
dtype: int64

In [43]:
#Creating a df for only games played:
played_games_df = steam_games_df[steam_games_df["Behavior_Name"] == "play"].copy()

display(played_games_df.head())
display(played_games_df["Behavior_Name"].value_counts())

,User_ID,Game_Title,Behavior_Name,Hours_Played,0
0,151603712,The Elder Scrolls V Skyrim,play,273.0,0
2,151603712,Fallout 4,play,87.0,0
4,151603712,Spore,play,14.9,0
6,151603712,Fallout New Vegas,play,12.1,0
8,151603712,Left 4 Dead 2,play,8.9,0


Behavior_Name
play    70489
Name: count, dtype: int64

In [44]:
#Checking how many users played games
played_games_df["User_ID"].nunique()

11350

In [45]:
#Checking how many users there were originally
steam_games_df["User_ID"].nunique()

12393

In [46]:
#Checking for nulls again
played_games_df.isnull().sum()

User_ID          0
Game_Title       0
Behavior_Name    0
Hours_Played     0
0                0
dtype: int64

In [54]:
print("Matrix:", played_games_df.shape)

#Rows, Columns

Matrix: (70489, 5)


In [48]:
played_games_df["Game_Title"].nunique()

3600

In [51]:
played_games_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 70489 entries, 0 to 199998
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   User_ID        70489 non-null  int64  
 1   Game_Title     70489 non-null  object 
 2   Behavior_Name  70489 non-null  object 
 3   Hours_Played   70489 non-null  float64
 4   0              70489 non-null  int64  
dtypes: float64(1), int64(2), object(2)
memory usage: 3.2+ MB


In [60]:
#Creating a matrix with pivot table instead of merge?:
play_matrix = played_games_df.pivot_table(index = "Game_Title", columns = "User_ID", values = "Hours_Played")
#Fill value 0 so 

play_matrix.head()

User_ID,5250,76767,86540,144736,181212,229911,298950,381543,547685,554278,...,309228590,309255941,309262440,309265377,309404240,309434439,309554670,309626088,309824202,309903146
Game_Title,,,,,,,,,,,,,,,,,,,,,
007 Legends,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0RBITALIS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1... 2... 3... KICK IT! (Drop That Beat Like an Ugly Baby),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10 Second Ninja,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"10,000,000",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [69]:
play_matrix.isnull().sum()

User_ID
5250         3594
76767        3580
86540        3585
144736       3599
181212       3598
             ... 
309434439    3599
309554670    3599
309626088    3599
309824202    3599
309903146    3599
Length: 11350, dtype: int64

In [70]:
play_matrix.notnull().sum()

User_ID
5250          6
76767        20
86540        15
144736        1
181212        2
             ..
309434439     1
309554670     1
309626088     1
309824202     1
309903146     1
Length: 11350, dtype: int64

In [74]:
play_matrix = play_matrix.fillna(0)

play_matrix.head()

User_ID,5250,76767,86540,144736,181212,229911,298950,381543,547685,554278,...,309228590,309255941,309262440,309265377,309404240,309434439,309554670,309626088,309824202,309903146
Game_Title,,,,,,,,,,,,,,,,,,,,,
007 Legends,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0RBITALIS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1... 2... 3... KICK IT! (Drop That Beat Like an Ugly Baby),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10 Second Ninja,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"10,000,000",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [129]:
#Initialize the query games we want to check:
query_games = ["Shelter", "Shelter 2", "Portal 2", "Assassin's Creed II", "Cat Goes Fishing", "Ori and the Blind Forest", "Octodad Dadliest Catch"] 

#Starts to loop through the queries:
for query_game_id in query_games:
    print("\n\nQuery Game: ", query_game_id)

    #Makes sure query is in the data
    if query_game_id not in play_matrix.index:
        print(query_game_id, "not found.")
        continue
        
    # Get the cosine distances for each game:
    cosine_distances = pairwise_distances(play_matrix,play_matrix.loc[[query_game_id]], metric = "cosine")[:,0]
    cosine_pairs = [(game_id,dist) for game_id, dist in zip(play_matrix.index, cosine_distances) if game_id != query_game_id]
    cosine_top10 = sorted(cosine_pairs, key=lambda x: x[1]) [:10]

    #Initialize jaccard list to get jaccard similarity metrics for each game:
    jaccard_list = []
    
    #Loop through to output the top 10 most similar games to the query:
    print("Top 10 Most Similar Games Using Jaccard Similarity")
    for game_id, dist in cosine_top10:
        #Gets the current row for the looped game:
        matrix_row = play_matrix.loc[game_id]
        #Gets total hours played for the current game:
        total_hours_played = matrix_row.sum()

        #Gets if user played the query game and counts:
        played_query = (play_matrix.loc[query_game_id] > 0).astype(int)
        #Gets if user played current looped game and counts:
        played_game = (play_matrix.loc[game_id] > 0).astype(int)
        
        #Gets the intersection and union for jaccard similarity:
        intersect = (played_query & played_game).sum()
        union = (played_query | played_game).sum()

        #Calculate the jaccard similarity:
        if union > 0:
            jaccard = intersect / union
        else:
            jaccard = 0.0

        #Count how many users played the current looped game for info
        players = (matrix_row > 0).sum()
        
        #Getting the average amount of hours played:
        if players > 0:
            average_hours = total_hours_played / players
        else:
            average_hours = 0

        #Putting the jaccard similarity into the list for games:
        jaccard_list.append({"game": game_id, "jaccard": jaccard, "dist": dist, "total_hours": total_hours_played, 
                                     "avg_hours": average_hours, "players": players})

    #Sorts by jaccard similarity 
    jaccard_similarities_sorted = sorted(jaccard_list, key=lambda x: x["jaccard"], reverse=True)

#print(f"\n{game_id:35}", f"dist={dist:.4f}", 
#    f"\nTotal Hours Played: {total_hours_played:7.1f} | ", 
    #f"Hours Played On Average: {average_hours:7.2f} | ", f"Count of Who Played: {players:5d} | ", f"Percentage of Who Also Played: {percent_overlap:.2f}%")

    for i in jaccard_similarities_sorted[:10]:
        print(f"{i['game']} |", f"\n    Jaccard={i['jaccard']:.2%} | ", f"| Cosine dist={i['dist']:.4f} | ", f"\n    Total Hours Played: {i['total_hours']:7.1f} | ",
              f"Hours Played On Average: {i['avg_hours']:7.2f} | ", f"Count of Who Played: {i['players']:5d} \n")





Query Game:  Shelter
Top 10 Most Similar Games Using Jaccard Similarity
Lume | 
    Jaccard=28.57% |  | Cosine dist=0.3860 |  
    Total Hours Played:     3.6 |  Hours Played On Average:    0.90 |  Count of Who Played:     4 

Blood of the Werewolf | 
    Jaccard=20.00% |  | Cosine dist=0.0260 |  
    Total Hours Played:     0.5 |  Hours Played On Average:    0.50 |  Count of Who Played:     1 

Coil | 
    Jaccard=20.00% |  | Cosine dist=0.0260 |  
    Total Hours Played:     0.3 |  Hours Played On Average:    0.30 |  Count of Who Played:     1 

Finding Teddy | 
    Jaccard=20.00% |  | Cosine dist=0.0260 |  
    Total Hours Played:     0.5 |  Hours Played On Average:    0.50 |  Count of Who Played:     1 

Rocketbirds Hardboiled Chicken | 
    Jaccard=18.18% |  | Cosine dist=0.3906 |  
    Total Hours Played:    15.8 |  Hours Played On Average:    1.98 |  Count of Who Played:     8 

World of Zoo | 
    Jaccard=16.67% |  | Cosine dist=0.0357 |  
    Total Hours Played:     2.4 |  H

query_games = ["Shelter", "Shelter 2", "Portal 2", "Papers, Please", "Dear Esther", "Finding Teddy", "Assassin's Creed II", "Cat Goes Fishing", "The Stanley Parable", "Ori and the Blind Forest", "Octodad Dadliest Catch"] 


some games I heard of before or seen

query_games = ["Shelter", "Shelter 2", "Portal 2", "Papers, Please", "Dear Esther", "Assassin's Creed II", "Assassin's Creed II", "Garry's Mod", "Cat Goes Fishing", "The Stanley Parable", "Ori and the Blind Forest", "Octodad Dadliest Catch", "Doctor Who The Eternity Clock", "Sonic & All-Stars Racing Transformed", 

didnthearthis
the cat and the coup 
heardthis
depression quest


Finding Teddy